In [ ]:
import cv2
import numpy as np
import os
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO

# --- CONFIGURATION ---
INPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content"
OUTPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/stress_tests"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONF_THRESHOLD = 0.50
FRAME_LIMIT = 150

# --- SYNTHETIC DEGRADATION FUNCTIONS ---

def apply_darkness(frame, severity=0.5):
    """
    Simulate low-light conditions
    severity: 0.0 (no change) to 1.0 (pitch black)
    """
    factor = 1.0 - severity
    darkened = (frame.astype(np.float32) * factor).clip(0, 255).astype(np.uint8)
    return darkened

def apply_fog(frame, severity=0.5):
    """
    Simulate atmospheric fog/haze
    severity: 0.0 (clear) to 1.0 (dense fog)
    """
    # Fog is a blend with white/gray atmospheric light
    atmospheric_light = np.ones_like(frame) * 200  # Grayish white
    fogged = cv2.addWeighted(frame, 1.0 - severity, 
                             atmospheric_light.astype(np.uint8), severity, 0)
    return fogged

def apply_noise(frame, severity=0.5):
    """
    Simulate sensor noise (Gaussian)
    severity: 0.0 (clean) to 1.0 (extreme noise)
    """
    noise = np.random.normal(0, severity * 50, frame.shape).astype(np.float32)
    noisy = (frame.astype(np.float32) + noise).clip(0, 255).astype(np.uint8)
    return noisy

def apply_motion_blur(frame, severity=0.5):
    """
    Simulate motion blur from fast movement
    severity: 0.0 (sharp) to 1.0 (severe blur)
    """
    kernel_size = int(severity * 15) + 1
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size-1)/2), :] = np.ones(kernel_size)
    kernel = kernel / kernel_size
    blurred = cv2.filter2D(frame, -1, kernel)
    return blurred

def apply_combined_degradation(frame, darkness=0, fog=0, noise=0, blur=0):
    """Apply multiple degradations in sequence"""
    degraded = frame.copy()
    if darkness > 0:
        degraded = apply_darkness(degraded, darkness)
    if fog > 0:
        degraded = apply_fog(degraded, fog)
    if noise > 0:
        degraded = apply_noise(degraded, noise)
    if blur > 0:
        degraded = apply_motion_blur(degraded, blur)
    return degraded

# --- ENHANCEMENT FUNCTIONS (from previous script) ---

def get_dark_channel(image, size=15):
    min_channel = np.min(image, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    return cv2.erode(min_channel, kernel)

def apply_dehaze(frame):
    img_f = frame.astype(np.float32) / 255.0
    dark = get_dark_channel(img_f, size=15)
    A = np.percentile(dark, 99)
    t = 1.0 - 0.95 * dark
    t = np.clip(t, 0.1, 1.0)
    J = (img_f - A) / cv2.merge([t, t, t]) + A
    J = np.clip(J, 0, 1)
    return (J * 255).astype(np.uint8)

def apply_clahe(frame):
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_adaptive_gamma(frame):
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y = img_yuv[:, :, 0]
    mean_bright = np.mean(y) + 1e-5
    gamma = np.log(128/255) / np.log(mean_bright/255)
    gamma = np.clip(gamma, 0.5, 2.5)
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    img_yuv[:, :, 0] = cv2.LUT(y, table)
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_bilateral_denoise(frame):
    """Bilateral filter - preserves edges while reducing noise"""
    return cv2.bilateralFilter(frame, 9, 75, 75)

def apply_nlm_denoise(frame):
    """Non-Local Means denoising - good for heavy noise"""
    return cv2.fastNlMeansDenoisingColored(frame, None, 10, 10, 7, 21)

# --- ENHANCEMENT PIPELINES ---

ENHANCEMENT_PIPELINES = {
    'none': lambda x: x,
    'dehaze_only': lambda x: apply_dehaze(x),
    'clahe_only': lambda x: apply_clahe(x),
    'gamma_clahe': lambda x: apply_clahe(apply_adaptive_gamma(x)),
    'dehaze_clahe': lambda x: apply_clahe(apply_dehaze(x)),
    'denoise_clahe': lambda x: apply_clahe(apply_bilateral_denoise(x)),
    'full_recovery': lambda x: apply_clahe(apply_adaptive_gamma(apply_bilateral_denoise(apply_dehaze(x)))),
}

# --- STRESS TEST SCENARIOS ---

STRESS_SCENARIOS = {
    'pristine': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    
    # Single degradations at different severities
    'mild_darkness': {'darkness': 0.3, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    'severe_darkness': {'darkness': 0.6, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    'extreme_darkness': {'darkness': 0.8, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    
    'mild_fog': {'darkness': 0.0, 'fog': 0.3, 'noise': 0.0, 'blur': 0.0},
    'dense_fog': {'darkness': 0.0, 'fog': 0.6, 'noise': 0.0, 'blur': 0.0},
    'extreme_fog': {'darkness': 0.0, 'fog': 0.8, 'noise': 0.0, 'blur': 0.0},
    
    'mild_noise': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.3, 'blur': 0.0},
    'heavy_noise': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.6, 'blur': 0.0},
    
    'mild_blur': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.0, 'blur': 0.3},
    'severe_blur': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.0, 'blur': 0.6},
    
    # Combined degradations (realistic scenarios)
    'night_fog': {'darkness': 0.5, 'fog': 0.4, 'noise': 0.2, 'blur': 0.0},
    'rainy_night': {'darkness': 0.6, 'fog': 0.3, 'noise': 0.3, 'blur': 0.2},
    'extreme_conditions': {'darkness': 0.7, 'fog': 0.5, 'noise': 0.4, 'blur': 0.3},
}



In [ ]:
# --- ANALYSIS ENGINE ---

def analyze_stressed_video(video_path, degradation_params, enhancement_method='none', 
                           frame_limit=FRAME_LIMIT, verbose=False):
    """
    Test video under stress conditions with optional enhancement
    """
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Initialize
    tracker = DeepSort(max_age=50)
    model = YOLO("yolov8n.pt")
    
    # Metrics
    track_ids = set()
    total_frames_tracked = 0
    confidences = []
    detections_per_frame = []
    brightness_values = []
    
    enhancement_func = ENHANCEMENT_PIPELINES[enhancement_method]
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx > frame_limit:
            break
        
        # 1. Apply degradation
        degraded = apply_combined_degradation(frame, **degradation_params)
        
        # 2. Apply enhancement
        processed = enhancement_func(degraded)
        
        # Track brightness
        gray = cv2.cvtColor(processed, cv2.COLOR_BGR2GRAY)
        brightness_values.append(np.mean(gray))
        
        # 3. Detect
        results = model(processed, verbose=False)
        
        dets = []
        for r in results:
            for box in r.boxes:
                if float(box.conf[0]) > CONF_THRESHOLD:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])
                    dets.append([[x1, y1, x2-x1, y2-y1], conf, int(box.cls[0])])
                    confidences.append(conf)
        
        detections_per_frame.append(len(dets))
        
        # 4. Track
        tracks = tracker.update_tracks(dets, frame=processed)
        for t in tracks:
            if t.is_confirmed():
                track_ids.add(t.track_id)
                total_frames_tracked += 1
        
        frame_idx += 1
    
    cap.release()
    
    # Compute metrics
    unique_ids = len(track_ids)
    avg_duration = total_frames_tracked / unique_ids if unique_ids > 0 else 0
    avg_confidence = np.mean(confidences) if confidences else 0
    avg_detections = np.mean(detections_per_frame)
    total_detections = sum(detections_per_frame)
    avg_brightness = np.mean(brightness_values)
    
    if verbose:
        print(f"    IDs:{unique_ids:3d} | Dur:{avg_duration:5.1f} | "
              f"Conf:{avg_confidence:.3f} | Det/F:{avg_detections:.2f} | "
              f"Total:{total_detections:4d} | Bright:{avg_brightness:.1f}")
    
    return {
        'Unique IDs': unique_ids,
        'Avg Duration': avg_duration,
        'Avg Confidence': avg_confidence,
        'Detections/Frame': avg_detections,
        'Total Detections': total_detections,
        'Avg Brightness': avg_brightness
    }



In [ ]:
print("="*80)
print("🔬 COMPREHENSIVE STRESS TEST SUITE")
print("="*80)
print(f"Test Scenarios: {len(STRESS_SCENARIOS)}")
print(f"Enhancement Methods: {len(ENHANCEMENT_PIPELINES)}")
print(f"Total Tests per Video: {len(STRESS_SCENARIOS) * len(ENHANCEMENT_PIPELINES)}")
print()

# Select test videos (use 2 for speed, increase for thoroughness)
video_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(('.avi', '.mp4'))]
video_files = [f for f in video_files if "result" not in f and "stress" not in f][:2]

print(f"Test Videos: {video_files}")
print()

all_results = []

for vid_idx, vid in enumerate(video_files):
    video_path = os.path.join(INPUT_DIR, vid)
    print(f"\n{'='*80}")
    print(f"📹 VIDEO {vid_idx+1}/{len(video_files)}: {vid}")
    print(f"{'='*80}\n")
    
    for scenario_name, deg_params in STRESS_SCENARIOS.items():
        print(f"🌧️  SCENARIO: {scenario_name:<20} (dark:{deg_params['darkness']:.1f} "
              f"fog:{deg_params['fog']:.1f} noise:{deg_params['noise']:.1f} "
              f"blur:{deg_params['blur']:.1f})")
        
        for enh_method in ENHANCEMENT_PIPELINES.keys():
            print(f"   Enhancement: {enh_method:<20}", end=" | ")
            
            result = analyze_stressed_video(
                video_path, 
                deg_params, 
                enh_method,
                frame_limit=FRAME_LIMIT,
                verbose=True
            )
            
            # Store results
            result['Video'] = vid
            result['Scenario'] = scenario_name
            result['Enhancement'] = enh_method
            result['Degradation_Darkness'] = deg_params['darkness']
            result['Degradation_Fog'] = deg_params['fog']
            result['Degradation_Noise'] = deg_params['noise']
            result['Degradation_Blur'] = deg_params['blur']
            
            all_results.append(result)

# Convert to DataFrame
df_stress = pd.DataFrame(all_results)

# Save raw results
df_stress.to_csv(os.path.join(OUTPUT_DIR, "stress_test_detailed.csv"), index=False)

print("\n" + "="*80)
print("📊 ANALYZING RESULTS...")
print("="*80)

# --- ANALYSIS 1: Enhancement Effectiveness by Degradation Type ---

print("\n1️⃣ ENHANCEMENT EFFECTIVENESS BY DEGRADATION TYPE")
print("-" * 80)

# Group by scenario and enhancement
pivot_confidence = df_stress.pivot_table(
    index='Scenario',
    columns='Enhancement',
    values='Avg Confidence',
    aggfunc='mean'
)

pivot_detections = df_stress.pivot_table(
    index='Scenario',
    columns='Enhancement',
    values='Total Detections',
    aggfunc='mean'
)

# Calculate improvement over 'none' baseline
improvement_confidence = pivot_confidence.div(pivot_confidence['none'], axis=0) - 1
improvement_detections = pivot_detections.div(pivot_detections['none'], axis=0) - 1

print("\nConfidence Improvement (% over no enhancement):")
print((improvement_confidence * 100).round(1).to_string())

print("\nDetection Count Improvement (% over no enhancement):")
print((improvement_detections * 100).round(1).to_string())

# --- ANALYSIS 2: Best Enhancement for Each Scenario ---

print("\n2️⃣ BEST ENHANCEMENT FOR EACH SCENARIO")
print("-" * 80)

best_by_scenario = []
for scenario in STRESS_SCENARIOS.keys():
    scenario_data = df_stress[df_stress['Scenario'] == scenario]
    best_conf = scenario_data.groupby('Enhancement')['Avg Confidence'].mean()
    best_method = best_conf.idxmax()
    best_value = best_conf.max()
    baseline_value = best_conf['none']
    improvement = ((best_value / baseline_value) - 1) * 100
    
    best_by_scenario.append({
        'Scenario': scenario,
        'Best Method': best_method,
        'Confidence': best_value,
        'Improvement': improvement
    })

df_best = pd.DataFrame(best_by_scenario)
print(df_best.to_string(index=False))

# --- ANALYSIS 3: Severity Analysis ---

print("\n3️⃣ PERFORMANCE vs DEGRADATION SEVERITY")
print("-" * 80)

# Darkness severity analysis
darkness_scenarios = ['pristine', 'mild_darkness', 'severe_darkness', 'extreme_darkness']
darkness_data = df_stress[df_stress['Scenario'].isin(darkness_scenarios)]

print("\nDARKNESS SEVERITY:")
for scenario in darkness_scenarios:
    data = darkness_data[darkness_data['Scenario'] == scenario]
    print(f"\n  {scenario}:")
    for method in ['none', 'dehaze_only', 'gamma_clahe', 'full_recovery']:
        method_data = data[data['Enhancement'] == method]
        if len(method_data) > 0:
            conf = method_data['Avg Confidence'].mean()
            dets = method_data['Total Detections'].mean()
            print(f"    {method:20s}: Conf={conf:.3f}, Detections={dets:.1f}")

# --- VISUALIZATION DASHBOARD ---

print("\n📈 Generating comprehensive visualizations...")

# Figure 1: Heatmap - Confidence by Scenario and Enhancement
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(pivot_confidence, annot=True, fmt='.3f', cmap='RdYlGn', 
            ax=axes[0], cbar_kws={'label': 'Avg Confidence'})
axes[0].set_title('Confidence Score by Scenario & Enhancement', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Enhancement Method', fontweight='bold')
axes[0].set_ylabel('Degradation Scenario', fontweight='bold')

sns.heatmap(improvement_confidence * 100, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, ax=axes[1], cbar_kws={'label': 'Improvement %'})
axes[1].set_title('Confidence Improvement Over Baseline (%)', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Enhancement Method', fontweight='bold')
axes[1].set_ylabel('Degradation Scenario', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "stress_heatmap_confidence.png"), dpi=150, bbox_inches='tight')
print("  ✅ Saved: stress_heatmap_confidence.png")

# Figure 2: Severity Curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Performance vs Degradation Severity', fontsize=16, fontweight='bold')

degradation_types = [
    ('darkness', ['pristine', 'mild_darkness', 'severe_darkness', 'extreme_darkness']),
    ('fog', ['pristine', 'mild_fog', 'dense_fog', 'extreme_fog']),
    ('noise', ['pristine', 'mild_noise', 'heavy_noise']),
    ('blur', ['pristine', 'mild_blur', 'severe_blur'])
]

for idx, (deg_type, scenarios) in enumerate(degradation_types):
    ax = axes[idx // 2, idx % 2]
    
    for method in ['none', 'dehaze_only', 'clahe_only', 'gamma_clahe', 'dehaze_clahe', 'full_recovery']:
        method_data = []
        severity_levels = []
        
        for scenario in scenarios:
            data = df_stress[(df_stress['Scenario'] == scenario) & 
                           (df_stress['Enhancement'] == method)]
            if len(data) > 0:
                method_data.append(data['Avg Confidence'].mean())
                severity_levels.append(STRESS_SCENARIOS[scenario][deg_type])
        
        if method_data:
            ax.plot(severity_levels, method_data, marker='o', linewidth=2, 
                   label=method, markersize=8)
    
    ax.set_xlabel(f'{deg_type.capitalize()} Severity', fontweight='bold')
    ax.set_ylabel('Avg Confidence', fontweight='bold')
    ax.set_title(f'Impact of {deg_type.capitalize()}', fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "severity_curves.png"), dpi=150, bbox_inches='tight')
print("  ✅ Saved: severity_curves.png")

# Figure 3: Best Enhancement by Scenario (Bar Chart)
fig, ax = plt.subplots(figsize=(14, 8))

scenarios_sorted = df_best.sort_values('Improvement', ascending=False)['Scenario'].tolist()
x_pos = np.arange(len(scenarios_sorted))

colors = []
for scenario in scenarios_sorted:
    improvement = df_best[df_best['Scenario'] == scenario]['Improvement'].values[0]
    if improvement > 10:
        colors.append('green')
    elif improvement > 5:
        colors.append('orange')
    else:
        colors.append('red')

bars = ax.bar(x_pos, 
              [df_best[df_best['Scenario'] == s]['Improvement'].values[0] for s in scenarios_sorted],
              color=colors, alpha=0.7)

ax.set_xticks(x_pos)
ax.set_xticklabels(scenarios_sorted, rotation=45, ha='right')
ax.set_ylabel('Confidence Improvement (%)', fontweight='bold')
ax.set_title('Best Enhancement Improvement by Scenario', fontweight='bold', fontsize=14)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    method = df_best[df_best['Scenario'] == scenarios_sorted[i]]['Best Method'].values[0]
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%\n{method}',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "best_enhancement_by_scenario.png"), dpi=150, bbox_inches='tight')
print("  ✅ Saved: best_enhancement_by_scenario.png")

# Figure 4: Detection Recovery Rate
fig, ax = plt.subplots(figsize=(12, 8))

scenarios_to_plot = ['severe_darkness', 'dense_fog', 'heavy_noise', 'night_fog', 
                     'rainy_night', 'extreme_conditions']
x = np.arange(len(scenarios_to_plot))
width = 0.12

methods_to_compare = ['none', 'dehaze_only', 'clahe_only', 'gamma_clahe', 
                      'dehaze_clahe', 'full_recovery']

for i, method in enumerate(methods_to_compare):
    detections = []
    for scenario in scenarios_to_plot:
        data = df_stress[(df_stress['Scenario'] == scenario) & 
                        (df_stress['Enhancement'] == method)]
        detections.append(data['Total Detections'].mean() if len(data) > 0 else 0)
    
    ax.bar(x + i*width, detections, width, label=method)

ax.set_xlabel('Degradation Scenario', fontweight='bold')
ax.set_ylabel('Total Detections', fontweight='bold')
ax.set_title('Detection Recovery Under Extreme Conditions', fontweight='bold', fontsize=14)
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(scenarios_to_plot, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "detection_recovery.png"), dpi=150, bbox_inches='tight')
print("  ✅ Saved: detection_recovery.png")

plt.show()

# --- FINAL SUMMARY ---

print("\n" + "="*80)
print("🎯 COMPREHENSIVE STRESS TEST SUMMARY")
print("="*80)

print("\n📌 TOP FINDINGS:")

# Find scenarios where enhancement helps most
significant_improvements = df_best[df_best['Improvement'] > 5].sort_values('Improvement', ascending=False)

if len(significant_improvements) > 0:
    print("\n✅ Enhancement is EFFECTIVE in these scenarios:")
    for _, row in significant_improvements.iterrows():
        print(f"   • {row['Scenario']:25s}: {row['Best Method']:20s} (+{row['Improvement']:.1f}%)")
else:
    print("\n⚠️  Enhancement shows minimal benefit across all tested scenarios")

# Find scenarios where enhancement doesn't help
minimal_improvements = df_best[df_best['Improvement'] < 2].sort_values('Improvement', ascending=False)

if len(minimal_improvements) > 0:
    print("\n❌ Enhancement is NOT EFFECTIVE in these scenarios:")
    for _, row in minimal_improvements.iterrows():
        print(f"   • {row['Scenario']:25s}: Best improvement only +{row['Improvement']:.1f}%")

# Overall best method
overall_best = df_stress.groupby('Enhancement').agg({
    'Avg Confidence': 'mean',
    'Total Detections': 'mean'
}).sort_values('Avg Confidence', ascending=False)

print("\n🏆 OVERALL BEST ENHANCEMENT METHODS (Across All Scenarios):")
print(overall_best.head(3).to_string())

print("\n" + "="*80)
print(f"✅ Stress test complete! All results saved to: {OUTPUT_DIR}/")
print("="*80)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Paths
INPUT_CSV = "stress_test_detailed.csv"
OUTPUT_DIR = "content/final_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load Data
df = pd.read_csv('content/processed/stress_tests/stress_test_detailed.csv')

# Define Key Metrics
metrics = ['Detections/Frame', 'Avg Confidence', 'Avg Duration']
scenarios = df['Scenario'].unique()

print(f"Analyzing Scenarios: {scenarios}")

# 1. THE "CROSSOVER" PLOT (The most important chart)
# This shows how Baseline dominates in 'Pristine' but (likely) fails in 'Rainy_Night'
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x='Scenario', y='Detections/Frame', hue='Enhancement', palette='viridis')
plt.title("The 'Crossover Effect': Where Enhancements Become Essential", fontsize=14, fontweight='bold')
plt.ylabel("Vehicle Detections Per Frame")
plt.xlabel("Environmental Condition")
plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "final_crossover_effect.png"))
plt.show()

# 2. EFFICIENCY ANALYSIS (Did we trade speed for nothing?)
# Filter for Pristine only to show the "Cost of Over-processing"
pristine_df = df[df['Scenario'] == 'pristine']
plt.figure(figsize=(10, 5))
sns.scatterplot(data=pristine_df, x='Total Detections', y='Avg Duration', hue='Enhancement', s=200, style='Enhancement')
plt.title("Pristine Conditions: Cost vs Benefit", fontsize=14, fontweight='bold')
plt.xlabel("Total Detections (Recall)")
plt.ylabel("Avg Track Duration (Stability)")
plt.grid(True, linestyle='--')
plt.savefig(os.path.join(OUTPUT_DIR, "pristine_tradeoff.png"))
plt.show()

# 3. GENERATE SUMMARY TABLE
summary = df.groupby(['Scenario', 'Enhancement'])[['Detections/Frame', 'Avg Confidence', 'Avg Duration']].mean()
print("\n=== FINAL CONCLUSIVE DATA ===")
print(summary)
summary.to_csv(os.path.join(OUTPUT_DIR, "final_conclusive_table.csv"))

In [ ]:
# Cell: ADAPTIVE PIPELINE WITH OPTIMAL THRESHOLD TUNING
# This implements smart conditional enhancement based on all our findings,
# then searches for optimal trigger thresholds

import cv2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO
from itertools import product

# --- CONFIGURATION ---
INPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content"
OUTPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/adaptive_optimal"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONF_THRESHOLD = 0.50
FRAME_LIMIT = 150

# --- SCENE ANALYSIS FUNCTIONS ---

def estimate_brightness(frame):
    """Compute average brightness (0-255)"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return np.mean(gray)

def estimate_fog_level(frame):
    """
    Estimate atmospheric fog using Dark Channel Prior
    Returns: 0.0 (clear) to 1.0 (dense fog)
    """
    # Dark channel = minimum across RGB channels
    dark_channel = np.min(frame, axis=2)
    # Average darkness (lower = more fog)
    fog_metric = 1.0 - (np.mean(dark_channel) / 255.0)
    return fog_metric

def estimate_contrast(frame):
    """Compute contrast as standard deviation of brightness"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return np.std(gray)

def estimate_noise_level(frame):
    """
    Estimate noise using Laplacian variance
    Higher = more noise/texture
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    return laplacian.var()

def analyze_scene(frame):
    """Comprehensive scene analysis"""
    return {
        'brightness': estimate_brightness(frame),
        'fog_level': estimate_fog_level(frame),
        'contrast': estimate_contrast(frame),
        'noise': estimate_noise_level(frame)
    }

# --- ENHANCEMENT BUILDING BLOCKS ---

def get_dark_channel(image, size=15):
    """Dark channel for dehazing"""
    min_channel = np.min(image, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    return cv2.erode(min_channel, kernel)

def apply_dehaze(frame, omega=0.95):
    """
    Dark Channel Prior Dehazing
    omega: haze removal strength (0.7-0.95)
    """
    img_f = frame.astype(np.float32) / 255.0
    dark = get_dark_channel(img_f, size=15)
    A = np.percentile(dark, 99)
    t = 1.0 - omega * dark
    t = np.clip(t, 0.1, 1.0)
    J = (img_f - A) / cv2.merge([t, t, t]) + A
    J = np.clip(J, 0, 1)
    return (J * 255).astype(np.uint8)

def apply_clahe(frame, clip_limit=2.0, tile_size=8):
    """
    CLAHE - Local contrast enhancement
    clip_limit: 1.0-4.0 (higher = more aggressive)
    tile_size: 4-16 (smaller = more local)
    """
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_adaptive_gamma(frame, target_brightness=128):
    """
    Adaptive Gamma Correction
    Automatically adjusts to target brightness
    """
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y = img_yuv[:, :, 0]
    mean_bright = np.mean(y) + 1e-5
    gamma = np.log(target_brightness/255) / np.log(mean_bright/255)
    gamma = np.clip(gamma, 0.5, 2.5)
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    img_yuv[:, :, 0] = cv2.LUT(y, table)
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_bilateral_denoise(frame, d=9, sigma_color=75, sigma_space=75):
    """Edge-preserving denoising"""
    return cv2.bilateralFilter(frame, d, sigma_color, sigma_space)

def apply_adaptive_histogram_equalization(frame):
    """
    Advanced hybrid contrast enhancement
    Combines global and local methods
    """
    # Convert to LAB color space (better than YUV for this)
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    # Apply CLAHE to L channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_clahe = clahe.apply(l)
    
    # Weighted blend: 70% CLAHE + 30% original (prevents over-enhancement)
    l_blended = cv2.addWeighted(l_clahe, 0.7, l, 0.3, 0)
    
    # Merge and convert back
    enhanced_lab = cv2.merge([l_blended, a, b])
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

# --- ADAPTIVE PIPELINE STRATEGIES ---

class AdaptivePipeline:
    """Base class for adaptive enhancement"""
    def __init__(self, name, description):
        self.name = name
        self.description = description
    
    def process(self, frame, scene_params):
        """Override this method"""
        raise NotImplementedError

class Strategy_Universal_CLAHE(AdaptivePipeline):
    """Apply CLAHE to everything (your question: what if we just CLAHE everything?)"""
    def __init__(self, clip_limit=2.0):
        super().__init__(
            "universal_clahe",
            f"CLAHE on all frames (clip={clip_limit})"
        )
        self.clip_limit = clip_limit
    
    def process(self, frame, scene_params):
        return apply_clahe(frame, clip_limit=self.clip_limit)

class Strategy_Fog_Aware(AdaptivePipeline):
    """Trigger dehaze when fog detected"""
    def __init__(self, fog_threshold=0.5, omega=0.95):
        super().__init__(
            "fog_aware",
            f"Dehaze if fog>{fog_threshold} (omega={omega})"
        )
        self.fog_threshold = fog_threshold
        self.omega = omega
    
    def process(self, frame, scene_params):
        if scene_params['fog_level'] > self.fog_threshold:
            return apply_dehaze(frame, omega=self.omega)
        return frame

class Strategy_Darkness_Aware(AdaptivePipeline):
    """Trigger CLAHE when dark"""
    def __init__(self, brightness_threshold=100, clip_limit=2.5):
        super().__init__(
            "darkness_aware",
            f"CLAHE if brightness<{brightness_threshold} (clip={clip_limit})"
        )
        self.brightness_threshold = brightness_threshold
        self.clip_limit = clip_limit
    
    def process(self, frame, scene_params):
        if scene_params['brightness'] < self.brightness_threshold:
            return apply_clahe(frame, clip_limit=self.clip_limit)
        return frame

class Strategy_Hybrid_Adaptive(AdaptivePipeline):
    """Advanced: Adjust enhancement based on multiple factors"""
    def __init__(self, fog_thresh=0.5, dark_thresh=100, fog_omega=0.95, clahe_clip=2.0):
        super().__init__(
            "hybrid_adaptive",
            f"Fog>{fog_thresh}:dehaze, Dark<{dark_thresh}:CLAHE"
        )
        self.fog_thresh = fog_thresh
        self.dark_thresh = dark_thresh
        self.fog_omega = fog_omega
        self.clahe_clip = clahe_clip
    
    def process(self, frame, scene_params):
        # Priority 1: Fog (most impactful based on tests)
        if scene_params['fog_level'] > self.fog_thresh:
            dehazed = apply_dehaze(frame, omega=self.fog_omega)
            # If also dark, add CLAHE
            if scene_params['brightness'] < self.dark_thresh:
                return apply_clahe(dehazed, clip_limit=self.clahe_clip)
            return dehazed
        
        # Priority 2: Darkness
        elif scene_params['brightness'] < self.dark_thresh:
            return apply_clahe(frame, clip_limit=self.clahe_clip)
        
        # Normal conditions: no processing
        return frame

class Strategy_Advanced_Hybrid_Contrast(AdaptivePipeline):
    """
    The "adaptive hybrid contrast method" - sophisticated approach
    Uses LAB color space + weighted blending + conditional gamma
    """
    def __init__(self, brightness_thresh=120, fog_thresh=0.4):
        super().__init__(
            "advanced_hybrid_contrast",
            "LAB-space adaptive histogram + conditional gamma"
        )
        self.brightness_thresh = brightness_thresh
        self.fog_thresh = fog_thresh
    
    def process(self, frame, scene_params):
        brightness = scene_params['brightness']
        fog = scene_params['fog_level']
        
        # If foggy: dehaze first
        if fog > self.fog_thresh:
            frame = apply_dehaze(frame, omega=0.9)
        
        # Adaptive contrast in LAB space
        enhanced = apply_adaptive_histogram_equalization(frame)
        
        # If still dark after CLAHE: apply gentle gamma
        if brightness < self.brightness_thresh:
            enhanced = apply_adaptive_gamma(enhanced, target_brightness=128)
        
        return enhanced

class Strategy_Baseline(AdaptivePipeline):
    """No processing - baseline"""
    def __init__(self):
        super().__init__("baseline", "No enhancement")
    
    def process(self, frame, scene_params):
        return frame

# --- DEGRADATION FUNCTIONS (from stress test) ---

def apply_darkness(frame, severity=0.5):
    factor = 1.0 - severity
    return (frame.astype(np.float32) * factor).clip(0, 255).astype(np.uint8)

def apply_fog(frame, severity=0.5):
    atmospheric_light = np.ones_like(frame) * 200
    return cv2.addWeighted(frame, 1.0 - severity, 
                          atmospheric_light.astype(np.uint8), severity, 0)

def apply_combined_degradation(frame, darkness=0, fog=0):
    degraded = frame.copy()
    if darkness > 0:
        degraded = apply_darkness(degraded, darkness)
    if fog > 0:
        degraded = apply_fog(degraded, fog)
    return degraded

# --- VIDEO ANALYSIS ENGINE ---

def analyze_video_adaptive(video_path, strategy, degradation_params=None, 
                          frame_limit=FRAME_LIMIT, verbose=False):
    """Test adaptive pipeline on video"""
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    tracker = DeepSort(max_age=50)
    model = YOLO("yolov8n.pt")
    
    track_ids = set()
    total_frames_tracked = 0
    confidences = []
    detections_per_frame = []
    
    # Track scene conditions
    scene_conditions = []
    enhancement_triggered = 0
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx > frame_limit:
            break
        
        # Apply degradation if specified (for testing)
        if degradation_params:
            frame = apply_combined_degradation(frame, **degradation_params)
        
        # Analyze scene
        scene_params = analyze_scene(frame)
        scene_conditions.append(scene_params)
        
        # Apply adaptive enhancement
        processed = strategy.process(frame, scene_params)
        
        # Track if enhancement was applied (check if frame changed)
        if not np.array_equal(frame, processed):
            enhancement_triggered += 1
        
        # Detect
        results = model(processed, verbose=False)
        
        dets = []
        for r in results:
            for box in r.boxes:
                if float(box.conf[0]) > CONF_THRESHOLD:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])
                    dets.append([[x1, y1, x2-x1, y2-y1], conf, int(box.cls[0])])
                    confidences.append(conf)
        
        detections_per_frame.append(len(dets))
        
        # Track
        tracks = tracker.update_tracks(dets, frame=processed)
        for t in tracks:
            if t.is_confirmed():
                track_ids.add(t.track_id)
                total_frames_tracked += 1
        
        frame_idx += 1
    
    cap.release()
    
    # Aggregate scene conditions
    avg_scene = {
        'brightness': np.mean([s['brightness'] for s in scene_conditions]),
        'fog_level': np.mean([s['fog_level'] for s in scene_conditions]),
        'contrast': np.mean([s['contrast'] for s in scene_conditions])
    }
    
    # Compute metrics
    unique_ids = len(track_ids)
    avg_duration = total_frames_tracked / unique_ids if unique_ids > 0 else 0
    avg_confidence = np.mean(confidences) if confidences else 0
    total_detections = sum(detections_per_frame)
    enhancement_rate = enhancement_triggered / frame_idx if frame_idx > 0 else 0
    
    if verbose:
        print(f"    IDs:{unique_ids:3d} | Dur:{avg_duration:5.1f} | "
              f"Conf:{avg_confidence:.3f} | Det:{total_detections:4d} | "
              f"Enh:{enhancement_rate*100:.0f}%")
    
    return {
        'Unique IDs': unique_ids,
        'Avg Duration': avg_duration,
        'Avg Confidence': avg_confidence,
        'Total Detections': total_detections,
        'Enhancement Rate': enhancement_rate,
        **avg_scene
    }

# --- THRESHOLD OPTIMIZATION ---

def optimize_thresholds(video_path, strategy_class, param_ranges, degradation_params=None):
    """
    Grid search to find optimal threshold parameters
    
    param_ranges: dict like {'fog_thresh': [0.3, 0.4, 0.5, 0.6, 0.7]}
    """
    print(f"\n🔍 OPTIMIZING THRESHOLDS FOR {strategy_class.__name__}")
    print(f"   Parameter ranges: {param_ranges}")
    print("-" * 80)
    
    # Generate all combinations
    param_names = list(param_ranges.keys())
    param_values = list(param_ranges.values())
    combinations = list(product(*param_values))
    
    results = []
    
    for combo in combinations:
        # Create strategy with these parameters
        params = dict(zip(param_names, combo))
        strategy = strategy_class(**params)
        
        # Test
        result = analyze_video_adaptive(video_path, strategy, degradation_params, verbose=False)
        result['params'] = str(params)
        result['strategy'] = strategy.name
        results.append(result)
        
        print(f"   {params} → Conf:{result['Avg Confidence']:.3f}, Det:{result['Total Detections']}")
    
    df = pd.DataFrame(results)
    
    # Find best
    best_idx = df['Avg Confidence'].idxmax()
    best_result = df.iloc[best_idx]
    
    print(f"\n✅ BEST: {best_result['params']}")
    print(f"   Confidence: {best_result['Avg Confidence']:.3f}")
    print(f"   Detections: {best_result['Total Detections']}")
    
    return df, best_result

# --- MAIN EXECUTION ---

print("="*80)
print("🚀 ADAPTIVE PIPELINE OPTIMIZATION & FINAL TESTING")
print("="*80)

# Select test video
video_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(('.avi', '.mp4'))]
video_files = [f for f in video_files if "result" not in f]
test_video = os.path.join(INPUT_DIR, video_files[0])

print(f"\nTest Video: {video_files[0]}")

# --- PART 1: TEST ALL STRATEGIES ON PRISTINE VIDEO ---

print("\n" + "="*80)
print("PART 1: STRATEGY COMPARISON ON PRISTINE VIDEO")
print("="*80)

strategies_to_test = [
    Strategy_Baseline(),
    Strategy_Universal_CLAHE(clip_limit=2.0),
    Strategy_Universal_CLAHE(clip_limit=3.0),  # More aggressive
    Strategy_Fog_Aware(fog_threshold=0.5),
    Strategy_Darkness_Aware(brightness_threshold=100),
    Strategy_Hybrid_Adaptive(fog_thresh=0.5, dark_thresh=100),
    Strategy_Advanced_Hybrid_Contrast(brightness_thresh=120, fog_thresh=0.4),
]

pristine_results = []
for strategy in strategies_to_test:
    print(f"\n{strategy.name}: {strategy.description}")
    result = analyze_video_adaptive(test_video, strategy, degradation_params=None, verbose=True)
    result['strategy'] = strategy.name
    pristine_results.append(result)

df_pristine = pd.DataFrame(pristine_results)

print("\n📊 PRISTINE VIDEO RESULTS:")
print(df_pristine[['strategy', 'Avg Confidence', 'Total Detections', 'Unique IDs', 
                   'Enhancement Rate', 'brightness', 'fog_level']].to_string(index=False))

# --- PART 2: THRESHOLD OPTIMIZATION FOR FOG SCENARIO ---

print("\n" + "="*80)
print("PART 2: THRESHOLD OPTIMIZATION - EXTREME FOG SCENARIO")
print("="*80)

# Test with 80% fog degradation
fog_degradation = {'darkness': 0.0, 'fog': 0.8}

print("\n🌫️  Testing Fog-Aware Strategy with different fog thresholds...")
df_fog_opt, best_fog = optimize_thresholds(
    test_video,
    Strategy_Fog_Aware,
    {
        'fog_threshold': [0.3, 0.4, 0.5, 0.6, 0.7],
        'omega': [0.85, 0.90, 0.95]
    },
    degradation_params=fog_degradation
)

# --- PART 3: THRESHOLD OPTIMIZATION FOR DARKNESS ---

print("\n" + "="*80)
print("PART 3: THRESHOLD OPTIMIZATION - SEVERE DARKNESS SCENARIO")
print("="*80)

darkness_degradation = {'darkness': 0.6, 'fog': 0.0}

print("\n🌑 Testing Darkness-Aware Strategy with different brightness thresholds...")
df_dark_opt, best_dark = optimize_thresholds(
    test_video,
    Strategy_Darkness_Aware,
    {
        'brightness_threshold': [80, 100, 120, 140],
        'clip_limit': [2.0, 2.5, 3.0]
    },
    degradation_params=darkness_degradation
)

# --- PART 4: TEST OPTIMIZED PARAMETERS ON ALL SCENARIOS ---

print("\n" + "="*80)
print("PART 4: COMPREHENSIVE TEST WITH OPTIMIZED PARAMETERS")
print("="*80)

# Extract best parameters
best_fog_params = eval(best_fog['params'])
best_dark_params = eval(best_dark['params'])

print(f"\nUsing optimized parameters:")
print(f"  Fog-Aware: {best_fog_params}")
print(f"  Darkness-Aware: {best_dark_params}")

# Create optimized strategies
optimized_strategies = [
    Strategy_Baseline(),
    Strategy_Fog_Aware(**best_fog_params),
    Strategy_Darkness_Aware(**best_dark_params),
    Strategy_Hybrid_Adaptive(
        fog_thresh=best_fog_params['fog_threshold'],
        dark_thresh=best_dark_params['brightness_threshold'],
        fog_omega=best_fog_params['omega'],
        clahe_clip=best_dark_params['clip_limit']
    ),
    Strategy_Advanced_Hybrid_Contrast(),
    Strategy_Universal_CLAHE(clip_limit=2.0),
]

# Test scenarios
test_scenarios = [
    ('pristine', {}),
    ('dense_fog', {'fog': 0.6}),
    ('extreme_fog', {'fog': 0.8}),
    ('severe_darkness', {'darkness': 0.6}),
    ('night_fog', {'darkness': 0.5, 'fog': 0.4}),
]

comprehensive_results = []

for scenario_name, deg_params in test_scenarios:
    print(f"\n📹 SCENARIO: {scenario_name}")
    print("-" * 80)
    
    for strategy in optimized_strategies:
        result = analyze_video_adaptive(test_video, strategy, 
                                       degradation_params=deg_params if deg_params else None,
                                       verbose=True)
        result['strategy'] = strategy.name
        result['scenario'] = scenario_name
        comprehensive_results.append(result)

df_comprehensive = pd.DataFrame(comprehensive_results)

# --- VISUALIZATION ---

print("\n📊 Generating comparison visualizations...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Optimized Adaptive Pipeline Performance', fontsize=16, fontweight='bold')

# Plot 1: Confidence by Scenario
ax = axes[0, 0]
pivot_conf = df_comprehensive.pivot_table(
    index='scenario',
    columns='strategy',
    values='Avg Confidence'
)
pivot_conf.plot(kind='bar', ax=ax, rot=45)
ax.set_title('Confidence by Scenario')
ax.set_ylabel('Avg Confidence')
ax.legend(loc='best', fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Plot 2: Detections by Scenario
ax = axes[0, 1]
pivot_det = df_comprehensive.pivot_table(
    index='scenario',
    columns='strategy',
    values='Total Detections'
)
pivot_det.plot(kind='bar', ax=ax, rot=45)
ax.set_title('Total Detections by Scenario')
ax.set_ylabel('Total Detections')
ax.legend(loc='best', fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Plot 3: Enhancement Trigger Rate
ax = axes[0, 2]
pivot_enh = df_comprehensive.pivot_table(
    index='scenario',
    columns='strategy',
    values='Enhancement Rate'
)
(pivot_enh * 100).plot(kind='bar', ax=ax, rot=45)
ax.set_title('Enhancement Trigger Rate (%)')
ax.set_ylabel('% Frames Enhanced')
ax.legend(loc='best', fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Plot 4: Fog Threshold Optimization
ax = axes[1, 0]
for omega in [0.85, 0.90, 0.95]:
    data = df_fog_opt[df_fog_opt['params'].str.contains(f"'omega': {omega}")]
    thresholds = [eval(p)['fog_threshold'] for p in data['params']]
    confs = data['Avg Confidence'].values
    ax.plot(thresholds, confs, marker='o', label=f'omega={omega}')
ax.set_xlabel('Fog Threshold')
ax.set_ylabel('Avg Confidence')
ax.set_title('Fog Threshold Optimization (80% Fog Scenario)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 5: Darkness Threshold Optimization
ax = axes[1, 1]
for clip in [2.0, 2.5, 3.0]:
    data = df_dark_opt[df_dark_opt['params'].str.contains(f"'clip_limit': {clip}")]
    thresholds = [eval(p)['brightness_threshold'] for p in data['params']]
    confs = data['Avg Confidence'].values
    ax.plot(thresholds, confs, marker='o', label=f'CLAHE clip={clip}')
ax.set_xlabel('Brightness Threshold')
ax.set_ylabel('Avg Confidence')
ax.set_title('Darkness Threshold Optimization (60% Dark)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 6: Strategy Ranking (Average Across All Scenarios)
ax = axes[1, 2]
strategy_avg = df_comprehensive.groupby('strategy').agg({
    'Avg Confidence': 'mean',
    'Total Detections': 'mean'
})
strategy_avg = strategy_avg.sort_values('Avg Confidence', ascending=True)
strategy_avg['Avg Confidence'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Avg Confidence (All Scenarios)')
ax.set_title('Overall Strategy Ranking')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "adaptive_pipeline_optimization.png"), 
            dpi=150, bbox_inches='tight')
print(f"  ✅ Saved: adaptive_pipeline_optimization.png")

# Save results
df_comprehensive.to_csv(os.path.join(OUTPUT_DIR, "comprehensive_results.csv"), index=False)
df_fog_opt.to_csv(os.path.join(OUTPUT_DIR, "fog_threshold_optimization.csv"), index=False)
df_dark_opt.to_csv(os.path.join(OUTPUT_DIR, "darkness_threshold_optimization.csv"), index=False)

# --- FINAL RECOMMENDATIONS ---

print("\n" + "="*80)
print("🎯 FINAL OPTIMIZED PIPELINE RECOMMENDATIONS")
print("="*80)

best_overall = df_comprehensive.groupby('strategy')['Avg Confidence'].mean().idxmax()
best_conf = df_comprehensive.groupby('strategy')['Avg Confidence'].mean().max()

print(f"\n🏆 BEST OVERALL STRATEGY: {best_overall}")
print(f"   Average Confidence: {best_conf:.3f}")

print("\n📋 OPTIMIZED PARAMETERS:")
print(f"   Fog Threshold: {best_fog_params['fog_threshold']:.2f}")
print(f"   Fog Omega: {best_fog_params['omega']:.2f}")
print(f"   Brightness Threshold: {best_dark_params['brightness_threshold']}")
print(f"   CLAHE Clip Limit: {best_dark_params['clip_limit']:.1f}")

print("\n💡 PRODUCTION-READY CODE:")
print("-" * 80)
print(f"""
# OPTIMAL ADAPTIVE PIPELINE (Based on Empirical Testing)

def production_adaptive_enhancement(frame):
    '''
    Optimized adaptive pipeline with data-driven thresholds
    Tested on {len(comprehensive_results)} scenarios
    '''
    # Scene analysis
    brightness = np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    dark_channel = np.min(frame, axis=2)
    fog_level = 1.0 - (np.mean(dark_channel) / 255.0)
    
    # Decision tree with optimized thresholds
    if fog_level > {best_fog_params['fog_threshold']:.2f}:
        # Fog detected: apply dehaze (proven +102% gain in extreme fog)
        img_f = frame.astype(np.float32) / 255.0
        dark = np.min(img_f, axis=2)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        dark = cv2.erode(dark, kernel)
        A = np.percentile(dark, 99)
        t = 1.0 - {best_fog_params['omega']:.2f} * dark
        t = np.clip(t, 0.1, 1.0)
        J = (img_f - A) / np.expand_dims(t, axis=2) + A
        J = np.clip(J, 0, 1)
        enhanced = (J * 255).astype(np.uint8)
        
        # If also dark, add CLAHE
        if brightness < {best_dark_params['brightness_threshold']}:
            img_yuv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2YUV)
            clahe = cv2.createCLAHE(clipLimit={best_dark_params['clip_limit']:.1f}, tileGridSize=(8,8))
            img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
            return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)
        return enhanced
    
    elif brightness < {best_dark_params['brightness_threshold']}:
        # Dark conditions: apply CLAHE (proven +7% gain)
        img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
        clahe = cv2.createCLAHE(clipLimit={best_dark_params['clip_limit']:.1f}, tileGridSize=(8,8))
        img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
        return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)
    
    else:
        # Normal conditions: no enhancement needed
        return frame

# Usage:
# processed_frame = production_adaptive_enhancement(original_frame)
""")

print("\n✅ Optimization complete! All results saved to:")
print(f"   {OUTPUT_DIR}/")

plt.show()

In [ ]:

# 1. Setup Data
data = {
    'Scenario': ['Pristine', 'Pristine', 
                 'Extreme Fog', 'Extreme Fog', 
                 'Severe Darkness', 'Severe Darkness'],
    'Pipeline': ['Baseline', 'Adaptive (Ours)', 
                 'Baseline', 'Adaptive (Ours)', 
                 'Baseline', 'Adaptive (Ours)'],
    'Confidence Score': [0.615, 0.615, 
                         0.311, 0.629, 
                         0.590, 0.631]
}

df = pd.DataFrame(data)

# Compute absolute improvement for annotation
improvement = {}
scenarios = ["Pristine", "Extreme Fog", "Severe Darkness"]

for s in scenarios:
    base = df[(df["Scenario"] == s) & (df["Pipeline"] == "Baseline")]["Confidence Score"].values[0]
    adapt = df[(df["Scenario"] == s) & (df["Pipeline"] == "Adaptive (Ours)")]["Confidence Score"].values[0]
    imp = adapt - base
    improvement[s] = imp

# 2. Plot
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

palette = {"Baseline": "#7f8c8d", "Adaptive (Ours)": "#27ae60"}

ax = sns.barplot(
    data=df, 
    x='Scenario', 
    y='Confidence Score', 
    hue='Pipeline', 
    palette=palette,
    edgecolor="black",
    linewidth=1.5
)

# 3. Add Annotations using absolute improvement
annotation_positions = {
    "Pristine": (-0.05, 0.63),
    "Extreme Fog": (1, 0.66),
    "Severe Darkness": (2.05, 0.66)
}

for s, (x, y) in annotation_positions.items():
    imp = improvement[s]
    label = f"+{imp:.3f}" if abs(imp) > 0.001 else "No Change"
    plt.text(
        x, y, label,
        ha='center', va='bottom',
        color='green' if imp > 0 else 'red',
        fontweight='bold', fontsize=10
    )

# 4. Styling
plt.title("Impact of Adaptive Processing vs. Baseline", fontsize=16, fontweight='bold')
plt.ylabel("YOLO Detection Confidence", fontsize=12, fontweight='bold')
plt.xlabel("Environmental Condition", fontsize=12, fontweight='bold')
plt.ylim(0, 0.8)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()
